In [1]:
pip install pandas langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 21.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [3]:
import pandas as pd
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from sentence_transformers import SentenceTransformer

In [4]:
df = pd.read_csv('/kaggle/input/datasets/rawanmoamed/cvs-data/final_24k_resumes_master.csv')
text_column = 'text' 
category_column = 'category'

In [5]:
def process_cv_row(row):
    text = str(row[text_column])
    
    phone_pattern = r'\d{3}\s\d{3}\s\d{4}'
    phone = re.findall(phone_pattern, text)
    phone = phone[0] if phone else "Unknown"
    
    clean_text = re.sub(phone_pattern, '', text)
    clean_text = re.sub(r'www\S+|http\S+|S+\s\w+\scom', '', clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    
    return clean_text, str(phone)

In [6]:
chunk_size = 500
chunk_overlap = int(chunk_size * 0.20)
custom_separators = ["\n\n", "experience", "summary", "education", "skills", ". ", "! ", "? ", "\n", " ", ""]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=custom_separators,
    is_separator_regex=False
)

In [7]:
all_processed_chunks = []

print(f"Processing {len(df)} resumes using the ID-based system...")
for index, row in df.iterrows():
  
    clean_text, phone = process_cv_row(row)
    

    row_chunks = text_splitter.split_text(clean_text)
    
    meta = {
        "candidate_id": f"CV_{index}", 
        "category": str(row[category_column]),
        "phone": phone
    }
    
    for i, chunk in enumerate(row_chunks):
        all_processed_chunks.append({
            "id": f"cv_{index}_chunk_{i}",
            "text": chunk,
            "metadata": meta
        })

Processing 24228 resumes using the ID-based system...


In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')
db_path = "/kaggle/working/my_vector_db"
client = chromadb.PersistentClient(path=db_path)

collection = client.get_or_create_collection(name="cv_matching")

if collection.count() == 0:
    documents = [item['text'] for item in all_processed_chunks]
    metadatas = [item['metadata'] for item in all_processed_chunks]
    ids = [item['id'] for item in all_processed_chunks]

    print("Generating Embeddings")
    embeddings = model.encode(documents, show_progress_bar=True).tolist()
    
    batch_size = 5000
    total_chunks = len(all_processed_chunks)
    
    print(f"Uploading to ChromaDB (Total chunks: {total_chunks})...")
    for i in range(0, total_chunks, batch_size):
        batch_end = min(i + batch_size, total_chunks)
        collection.add(
            embeddings=embeddings[i:batch_end],
            documents=documents[i:batch_end],
            metadatas=metadatas[i:batch_end],
            ids=ids[i:batch_end]
        )
        print(f" Uploaded batch from {i} to {batch_end}")
else:
    print(f"Database already contains {collection.count()} items.")
    

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating Embeddings


Batches:   0%|          | 0/9147 [00:00<?, ?it/s]

Uploading to ChromaDB (Total chunks: 292678)...
 Uploaded batch from 0 to 5000
 Uploaded batch from 5000 to 10000
 Uploaded batch from 10000 to 15000
 Uploaded batch from 15000 to 20000
 Uploaded batch from 20000 to 25000
 Uploaded batch from 25000 to 30000
 Uploaded batch from 30000 to 35000
 Uploaded batch from 35000 to 40000
 Uploaded batch from 40000 to 45000
 Uploaded batch from 45000 to 50000
 Uploaded batch from 50000 to 55000
 Uploaded batch from 55000 to 60000
 Uploaded batch from 60000 to 65000
 Uploaded batch from 65000 to 70000
 Uploaded batch from 70000 to 75000
 Uploaded batch from 75000 to 80000
 Uploaded batch from 80000 to 85000
 Uploaded batch from 85000 to 90000
 Uploaded batch from 90000 to 95000
 Uploaded batch from 95000 to 100000
 Uploaded batch from 100000 to 105000
 Uploaded batch from 105000 to 110000
 Uploaded batch from 110000 to 115000
 Uploaded batch from 115000 to 120000
 Uploaded batch from 120000 to 125000
 Uploaded batch from 125000 to 130000
 Uploaded

In [9]:
job_description = "Seeking a Senior Consultant to provide expert advice and strategic solutions. Must have experience in business analysis and project management."
query_vector = model.encode([job_description]).tolist()

results = collection.query(
    query_embeddings=query_vector,
    n_results=3
)

print("\n--- Search Results (Anonymous IDs) ---")
for i in range(len(results['documents'][0])):
    print(f"\nResult #{i+1}")
    print(f"Similarity Score: {results['distances'][0][i]:.4f}")
    print(f"Candidate ID: {results['metadatas'][0][i]['candidate_id']}") 
    print(f"Category: {results['metadatas'][0][i]['category']}")
    print(f"Phone: {results['metadatas'][0][i]['phone']}") 
    print(f"Snippet: {results['documents'][0][i][:150]}...")


--- Search Results (Anonymous IDs) ---

Result #1
Similarity Score: 0.4865
Candidate ID: CV_19784
Category: Consultant
Phone: Unknown
Snippet: senior consultant...

Result #2
Similarity Score: 0.5472
Candidate ID: CV_21914
Category: Consultant
Phone: 241 603 2068
Snippet: anagement consultant sample resume 695 wynd rod westminster co 0618 jacob deena com management consultant with over 12 years of experience in analysis...

Result #3
Similarity Score: 0.5842
Candidate ID: CV_19836
Category: Consultant
Phone: Unknown
Snippet: consultant summary job title with more than number years of experience planning developing and implementing program or process experienced manager wit...
